# Router Component - Hop Count Classification

**Model folder:** `components/router/models/qwen3_1_7b`

**Component:** Router (Stage A)

**Execution:** Kaggle or Colab

## 1. Environment + Versions

In [ ]:
import sys
import json
import random
import re
from pathlib import Path
from datetime import datetime

import torch
import transformers

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Config

In [ ]:
MODEL_NAME = "qwen3_1_7b"
MODEL_ID = "Qwen/Qwen3-1.7B"

CONFIG = {
    "seed": 42,
    "model_id": MODEL_ID,
    "model_name": MODEL_NAME,
    "data_path": "/kaggle/input/metaqa-questions/refined_1hop.txt",
    "data_path_2hop": "/kaggle/input/metaqa-questions/refined_2hop.txt",
    "data_path_3hop": "/kaggle/input/metaqa-questions/refined_3hop.txt",
    "output_dir": Path("/kaggle/working/runs/router"),
    "run_id": datetime.now().strftime("%Y%m%d_%H%M%S"),
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "max_new_tokens": 64,
    "temperature": 0.0,
    "top_p": 0.0,
    "top_k": 20,
    "do_sample": False,
    "max_thinking_chars": 1200,
    "approach": "few_shot_direct",
}

# Set seed for reproducibility
random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

print(f"Config: {json.dumps(CONFIG, indent=2, default=str)}")
print(f"Seed: {CONFIG['seed']}")
print(f"Approach: {CONFIG['approach']}")



In [ ]:
# =========================
# Router mode: "eval" or "debug"
# =========================
MODE = "eval"  # change to "debug" when you want to inspect thinking

def apply_router_mode(CONFIG, mode: str):
    if mode == "eval":
        # Fast + stable router settings
        CONFIG["enable_thinking"] = False      # best for speed + format reliability
        CONFIG["show_thinking"] = False
        CONFIG["max_new_tokens"] = 32
        CONFIG["temperature"] = 0.0
        CONFIG["top_p"] = 1.0
        CONFIG["top_k"] = 0
        CONFIG["do_sample"] = False
    else:
        # Debug: allow thinking, but keep deterministic to avoid format drift
        CONFIG["enable_thinking"] = True
        CONFIG["show_thinking"] = True
        CONFIG["max_new_tokens"] = 128
        CONFIG["temperature"] = 0.0
        CONFIG["top_p"] = 1.0
        CONFIG["top_k"] = 0
        CONFIG["do_sample"] = False

    return CONFIG

CONFIG = apply_router_mode(CONFIG, MODE)
print("Router MODE:", MODE)
print("Router config snapshot:", {k: CONFIG[k] for k in [
    "enable_thinking", "show_thinking", "max_new_tokens",
    "temperature", "top_p", "top_k", "do_sample"
]})


## 3. Data Loading

In [ ]:
def load_questions(file_path: str):
    """Load questions from a text file (one per line)."""
    path = Path(file_path)
    if not path.exists():
        print(f"Warning: {file_path} not found. Using placeholder data.")
        return []

    with open(path, "r", encoding="utf-8") as f:
        questions = [line.strip() for line in f if line.strip()]
    return questions


# Load test questions (update paths when MetaQA dataset is uploaded)
questions_1hop = load_questions(CONFIG["data_path"])
questions_2hop = load_questions(CONFIG["data_path_2hop"])
questions_3hop = load_questions(CONFIG["data_path_3hop"])

print(f"Loaded {len(questions_1hop)} 1-hop questions")
print(f"Loaded {len(questions_2hop)} 2-hop questions")
print(f"Loaded {len(questions_3hop)} 3-hop questions")

# Example questions for testing if files not found
if not questions_1hop:
    questions_1hop = ["What genre is Inception?"]
if not questions_2hop:
    questions_2hop = ["Who directed movies starring Brad Pitt?"]
if not questions_3hop:
    questions_3hop = ["What languages are films that share directors with The Matrix?"]

## 4. Model Setup

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"Loading model: {CONFIG['model_id']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_id"])

if CONFIG["device"] == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        CONFIG["model_id"],
        torch_dtype=torch.float16,
        device_map="auto",
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        CONFIG["model_id"],
        torch_dtype=torch.float32,
    ).to(CONFIG["device"])

model.eval()
MODEL_DEVICE = next(model.parameters()).device
print("Model primary device:", MODEL_DEVICE)

print("Model loaded successfully")

## 5. Router Function

In [ ]:
# =========================
# Prompt template (Step-based) – reduces 2-hop->3-hop bias
# =========================
PROMPT_TEMPLATE = """You are a hop-count router. Determine how many hops (linked lookups) are required.

Definition:
- 1-hop: answerable with ONE lookup/fact.
- 2-hop: requires TWO linked lookups.
- 3-hop: requires THREE linked lookups.
Important: Count LOOKUPS/RELATIONS, not the number of words or entities.

Output rules:
- Provide brief reasoning (step-based).
- Final output must be exactly: Output: 1 OR Output: 2 OR Output: 3

Examples:

Example 1:
Question: What genre is Inception?
Reasoning: Step 1: Find genre of Inception.
Output: 1

Example 2:
Question: Who directed movies starring Brad Pitt?
Reasoning: Step 1: Find movies starring Brad Pitt. Step 2: Find directors of those movies.
Output: 2

Example 3:
Question: The director of Avatar directed which other movies?
Reasoning: Step 1: Find director of Avatar. Step 2: Find other movies directed by that person.
Output: 2

# MetaQA-style 2-hop calibration examples (these reduce 2->3 errors)
Example 4:
Question: Who co-wrote films with Sam Shepard?
Reasoning: Step 1: Find films Sam Shepard co-wrote. Step 2: Find co-writers of those films.
Output: 2

Example 5:
Question: What are the genres of the movies written by Al Pacino?
Reasoning: Step 1: Find movies written by Al Pacino. Step 2: Find genres of those movies.
Output: 2

Example 6:
Question: Which movies have the same actor of John Krasinski?
Reasoning: Step 1: Find an actor in a John Krasinski movie. Step 2: Find other movies featuring that actor.
Output: 2

# 3-hop examples
Example 7:
Question: What languages are films that share directors with The Matrix?
Reasoning: Step 1: Find director(s) of The Matrix. Step 2: Find other films by those director(s). Step 3: Find languages of those films.
Output: 3

Example 8:
Question: Who acted in movies whose directors also directed Inception?
Reasoning: Step 1: Find director of Inception. Step 2: Find other movies by that director. Step 3: Find actors in those movies.
Output: 3

Now classify the question below.
Question: {question}
Reasoning:
Output:"""

def build_prompt(question: str) -> str:
    return PROMPT_TEMPLATE.format(question=question)





THINK_RE = re.compile(r"<think>(.*?)</think>", re.DOTALL | re.IGNORECASE)

def extract_think_and_content(raw: str):
    m = THINK_RE.search(raw)
    if not m:
        return None, raw.strip()
    thinking = m.group(1).strip()
    content = THINK_RE.sub("", raw).strip()
    return thinking, content


def classify_hop_count(question: str, model, tokenizer, device) -> int:
    prompt = build_prompt(question)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=CONFIG["enable_thinking"],
    )
    inputs = tokenizer([text], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            temperature=CONFIG["temperature"],
            top_p=CONFIG["top_p"],
            top_k=CONFIG["top_k"],
            do_sample=CONFIG["do_sample"],
            pad_token_id=tokenizer.pad_token_id,
        )

    output_ids = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    thinking_content, response = extract_think_and_content(raw)

    # Optional: show thinking only in debug mode
    if CONFIG.get("show_thinking") and thinking_content:
        preview = thinking_content[: CONFIG.get("max_thinking_chars", 800)]
        print("\n[thinking]\n" + preview + ("..." if len(thinking_content) > len(preview) else ""))

    # If model didn't output anything outside <think>, parse from thinking
    if (not response) and thinking_content:
        response = thinking_content

    text_out = response.strip()

    # ---- Parse hop count ----
    # Strongest: explicit Output: X
    m = re.search(r"output\s*:\s*([123])", text_out, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))

    # Next: any standalone 1/2/3 on a line (common if model prints just "2")
    lines = [line.strip() for line in text_out.splitlines() if line.strip()]
    for line in reversed(lines):
        if line in {"1", "2", "3"}:
            return int(line)

    # Next: 'answer' line fallback
    for line in reversed(lines):
        if "answer" in line.lower():
            m = re.search(r"\b([123])\b", line)
            if m:
                return int(m.group(1))

    # Last resort: any standalone digit 1/2/3 in text
    matches = re.findall(r"\b([123])\b", text_out)
    if matches:
        return int(matches[-1])

    # Final fallback
    # (should be rare after deterministic decoding + Output constraint)
    # Keep it silent in eval, noisy in debug
    if MODE == "debug":
        print(f"Warning: No hop count parsed. Raw='{raw[:200]}' -> defaulting to 2.")
    return 2
    

#Test Some outputs


In [ ]:
def generate_raw_response(question: str, model, tokenizer, device, keep_think: bool = True) -> str:
    """
    Generate the model's raw text output for the router prompt (optionally including <think>...</think>).

    Requires:
      - build_prompt(question)
      - THINK_RE (compiled regex for <think>...</think>)
      - CONFIG with generation params + enable_thinking
    """
    prompt = build_prompt(question)

    # Ensure pad_token exists
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Use chat template (recommended for Qwen3 chat/instruct behavior)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=CONFIG.get("enable_thinking", False),
    )

    inputs = tokenizer([text], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=CONFIG.get("max_new_tokens", 64),
            temperature=CONFIG.get("temperature", 0.0),
            top_p=CONFIG.get("top_p", 1.0),
            top_k=CONFIG.get("top_k", 0),
            do_sample=CONFIG.get("do_sample", False),
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only newly generated tokens
    output_ids = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    if not keep_think:
        # Remove <think>...</think> block if present
        raw = THINK_RE.sub("", raw).strip()

    return raw


In [ ]:
def test_random_samples_each_hop(
    questions_1hop,
    questions_2hop,
    questions_3hop,
    model,
    tokenizer,
    device,
    n=10,
    show_raw_on_mismatch=False,
    keep_think=True,
    seed=None
):
    """
    Randomly sample n questions from each hop list (1/2/3), run router, and print results.

    Params:
      show_raw_on_mismatch: if True, prints the raw generation only when prediction is wrong.
                            Requires generate_raw_response(...) to exist.
      keep_think: if True, raw output includes <think>...</think> (if model emits it).
      seed: optional int for reproducible sampling.
    """
    if seed is not None:
        rnd = random.Random(seed)
    else:
        rnd = random

    sets = [
        ("1-hop", questions_1hop, 1),
        ("2-hop", questions_2hop, 2),
        ("3-hop", questions_3hop, 3),
    ]

    all_correct = 0
    all_total = 0

    for label, qs, expected in sets:
        if not qs:
            print(f"\n=== {label} ===")
            print("No questions loaded.")
            continue

        sample = rnd.sample(qs, min(n, len(qs)))

        print(f"\n=== {label} | sampled {len(sample)} ===")
        correct = 0

        for i, q in enumerate(sample, 1):
            pred = classify_hop_count(q, model, tokenizer, device)
            ok = (pred == expected)
            correct += int(ok)
            all_correct += int(ok)
            all_total += 1

            status = "✅" if ok else "❌"
            print(f"{status} [{i:02d}] expected={expected} pred={pred} | {q}")

            if (not ok) and show_raw_on_mismatch:
                if "generate_raw_response" not in globals():
                    print("   (raw output requested but generate_raw_response() is not defined)")
                else:
                    raw = generate_raw_response(q, model, tokenizer, device, keep_think=keep_think)
                    print("   --- RAW OUTPUT START ---")
                    print(raw)
                    print("   --- RAW OUTPUT END ---")

        acc = correct / len(sample)
        print(f"{label} accuracy on sample: {acc:.3f} ({correct}/{len(sample)})")

    if all_total > 0:
        overall = all_correct / all_total
        print(f"\n=== Overall sample accuracy: {overall:.3f} ({all_correct}/{all_total}) ===")


In [ ]:
MODEL_DEVICE = next(model.parameters()).device

test_random_samples_each_hop(
    questions_1hop, questions_2hop, questions_3hop,
    model, tokenizer, MODEL_DEVICE,
    n=10,
    seed=42,
    keep_think=True,
    show_raw_only_on_mismatch=True
)


## 6. Run / Inference

In [ ]:
def classify_batch(questions, model, tokenizer, device: str, show_progress: bool = True):
    results = []
    for i, q in enumerate(questions):
        if show_progress and (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{len(questions)}...")
        hop = classify_hop_count(q, model, tokenizer, device)
        results.append(hop)
    return results


all_questions = []
all_expected = []

if questions_1hop:
    all_questions.extend(questions_1hop)
    all_expected.extend([1] * len(questions_1hop))
if questions_2hop:
    all_questions.extend(questions_2hop)
    all_expected.extend([2] * len(questions_2hop))
if questions_3hop:
    all_questions.extend(questions_3hop)
    all_expected.extend([3] * len(questions_3hop))

print(f"Running inference on {len(all_questions)} questions...")
print(f"  1-hop: {len(questions_1hop)}, 2-hop: {len(questions_2hop)}, 3-hop: {len(questions_3hop)}")
print()

predictions = classify_batch(all_questions, model, tokenizer, CONFIG["device"])

## 7. Evaluation + Metrics

In [ ]:
# Calculate metrics
correct = sum(1 for p, e in zip(predictions, all_expected) if p == e)
accuracy = correct / len(predictions) if predictions else 0.0

hop_1_correct = sum(1 for p, e in zip(predictions, all_expected) if e == 1 and p == e)
hop_1_total = sum(1 for e in all_expected if e == 1)
hop_1_acc = hop_1_correct / hop_1_total if hop_1_total > 0 else 0.0

hop_2_correct = sum(1 for p, e in zip(predictions, all_expected) if e == 2 and p == e)
hop_2_total = sum(1 for e in all_expected if e == 2)
hop_2_acc = hop_2_correct / hop_2_total if hop_2_total > 0 else 0.0

hop_3_correct = sum(1 for p, e in zip(predictions, all_expected) if e == 3 and p == e)
hop_3_total = sum(1 for e in all_expected if e == 3)
hop_3_acc = hop_3_correct / hop_3_total if hop_3_total > 0 else 0.0

from collections import Counter
confusion = Counter()
for p, e in zip(predictions, all_expected):
    confusion[(e, p)] += 1

metrics = {
    "overall_accuracy": accuracy,
    "total_questions": len(predictions),
    "correct_predictions": correct,
    "hop_1_accuracy": hop_1_acc,
    "hop_1_total": hop_1_total,
    "hop_2_accuracy": hop_2_acc,
    "hop_2_total": hop_2_total,
    "hop_3_accuracy": hop_3_acc,
    "hop_3_total": hop_3_total,
    "seed": CONFIG["seed"],
    "model": CONFIG["model_id"],
    "run_id": CONFIG["run_id"],
}

print("\n" + "=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"Overall Accuracy: {accuracy:.4f} ({correct}/{len(predictions)})")
print(f"1-hop Accuracy:   {hop_1_acc:.4f} ({hop_1_correct}/{hop_1_total})")
print(f"2-hop Accuracy:   {hop_2_acc:.4f} ({hop_2_correct}/{hop_2_total})")
print(f"3-hop Accuracy:   {hop_3_acc:.4f} ({hop_3_correct}/{hop_3_total})")

print("\n--- Confusion Matrix (Expected -> Predicted) ---")
print("     Pred->   1    2    3")
for expected in [1, 2, 3]:
    row = f"True {expected}:"
    for predicted in [1, 2, 3]:
        count = confusion.get((expected, predicted), 0)
        row += f" {count:4d}"
    print(row)

## 8. Save Artifacts

In [ ]:
output_dir = CONFIG["output_dir"] / CONFIG["run_id"]
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

with open(output_dir / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

notes = f"""Router Component Run - {CONFIG['run_id']}

Model: {CONFIG['model_id']}
Seed: {CONFIG['seed']}

Results:
- Overall Accuracy: {metrics['overall_accuracy']:.4f}
- 1-hop Accuracy: {metrics['hop_1_accuracy']:.4f} ({metrics['hop_1_total']} questions)
- 2-hop Accuracy: {metrics['hop_2_accuracy']:.4f} ({metrics['hop_2_total']} questions)
- 3-hop Accuracy: {metrics['hop_3_accuracy']:.4f} ({metrics['hop_3_total']} questions)

What to check:
- Review per-hop accuracy to identify which hop counts are harder to classify
- Check detailed_results.json for specific misclassification patterns
"""

with open(output_dir / "notes.md", "w") as f:
    f.write(notes)

detailed_results = [
    {
        "question": q,
        "expected_hop": e,
        "predicted_hop": p,
        "correct": p == e,
    }
    for q, e, p in zip(all_questions, all_expected, predictions)
]

with open(output_dir / "detailed_results.json", "w") as f:
    json.dump(detailed_results, f, indent=2, ensure_ascii=False)

output_timestamp = datetime.now().isoformat()
model_outputs = [
    {
        "question": q,
        "model_answer": p,
        "model_name": CONFIG["model_id"],
        "run_id": CONFIG["run_id"],
        "timestamp": output_timestamp,
    }
    for q, p in zip(all_questions, predictions)
]

with open(output_dir / "model_outputs.json", "w") as f:
    json.dump(model_outputs, f, indent=2, ensure_ascii=False)

confusion_data = {
    "matrix": {f"true_{e}_pred_{p}": confusion.get((e, p), 0) for e in [1, 2, 3] for p in [1, 2, 3]},
    "labels": [1, 2, 3],
}
with open(output_dir / "confusion.json", "w") as f:
    json.dump(confusion_data, f, indent=2)

print(f"\nArtifacts saved to: {output_dir}")
print("- metrics.json")
print("- config.json")
print("- notes.md")
print("- detailed_results.json")
print("- model_outputs.json")
print("- confusion.json")